# Chronic Kidney Disease (CKD): Regression and Classification

**Objective:** Explore an end-to-end supervised machine learning workflow using a small healthcare dataset.

- **Regression:** Predict an *approximate* glomerular filtration rate (GFR) represented by an interval-derived numerical proxy.
- **Classification:** Predict the dataset's `class` label (`ckd` vs. `notckd`).

This notebook uses **Linear Regression** and **Logistic Regression** with train-only preprocessing, clear evaluation, and sanity checks. It is a **learning project, not a medical diagnostic tool**.

## 1. Dataset and reproducibility

**Source:** [UCI — Risk Factor Prediction of Chronic Kidney Disease (dataset 857)](https://archive.ics.uci.edu/dataset/857/risk+factor+prediction+of+chronic+kidney+disease).

The dataset contains **200 records** and **28 feature columns** plus the `class` target. The original `grf` column (spelled this way in the dataset) contains intervals and open-ended categories, not measured exact GFR values. Several laboratory measurements are also encoded as string categories.

The notebook downloads the data through `ucimlrepo`. Run with internet access (e.g. Google Colab). Using `random_state=42` makes the splits reproducible on a given dataset and software environment.

*Citation:* Islam, M. & Akter, S. (2020). *Risk Factor Prediction of Chronic Kidney Disease*. UCI Machine Learning Repository. https://doi.org/10.24432/C5WP64 (CC BY 4.0).

### Install dataset loader (Colab)

In [ ]:
%pip -q install ucimlrepo

### Import libraries

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay,
)

RANDOM_STATE = 42

### Load dataset

In [ ]:
ckd_dataset = fetch_ucirepo(id=857)
X_original = ckd_dataset.data.features.copy()
y_original = ckd_dataset.data.targets.copy()

df = pd.concat([X_original, y_original], axis=1)
print('Original shape:', df.shape)
print('Duplicate rows:', df.duplicated().sum())

df = df.drop_duplicates().copy()
print('Shape after deduplication:', df.shape)
display(df.head())

## 2. Inspect the dataset

Check feature types, class distribution, and null counts **before** preprocessing. A reported null count of zero does not guarantee clean data: some invalid values may be stored as text (e.g. `p` in the GFR column). Values such as `1-Jan` in `al` also warrant source verification; this notebook does not silently reinterpret them.

In [ ]:
print('Columns:', df.columns.tolist())
print('\nData types:')
print(df.dtypes.value_counts())
print('\nMissing values:')
print(df.isna().sum().sort_values(ascending=False).head(10))
print('\nClass distribution:')
print(df['class'].value_counts(dropna=False))
print('\nOriginal GFR values:')
print(df['grf'].value_counts(dropna=False))

In [ ]:
df['class'].value_counts().plot(kind='bar', edgecolor='black')
plt.title('CKD class distribution')
plt.ylabel('Records')
plt.xticks(rotation=0)
plt.show()

## 3. Convert GFR intervals for an educational regression target

Most `grf` values are ranges such as `51.7832 - 76.949`. For a **closed interval**, use its midpoint. For `< 26.6175`, use the midpoint of `[0, 26.6175)` **assuming a non-negative GFR**. An unbounded interval such as `≥ 227.944` has no justified midpoint, so we **exclude it from regression**, not classification. Unrecognized entries (e.g. `p`) are also excluded **only from regression**.

**Limitations:** These values are proxies, not true measured GFR. In particular, the `[0, upper)` assumption can introduce sizeable error for low GFR. RMSE, MAE, and R² here describe prediction of *these proxies*, not clinical precision. Exclusions can introduce selection bias.

In [ ]:
def gfr_to_approx(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()
    number = r'(\d+(?:\.\d+)?)'

    # Example: 51.7832 - 76.949
    match = re.fullmatch(number + r'\s*-\s*' + number, value)
    if match:
        lower = float(match.group(1))
        upper = float(match.group(2))
        return (lower + upper) / 2 if lower <= upper else np.nan

    # Example: < 26.6175; assumption: GFR cannot be negative.
    match = re.fullmatch(r'<\s*' + number, value)
    if match:
        return float(match.group(1)) / 2

    # Open upper bound (>= ...) or invalid text: midpoint unknown.
    return np.nan

# Parser checks: includes edge cases from the UCI dataset.
assert np.isclose(gfr_to_approx('26.6175 - 51.7832'), 39.20035)
assert np.isclose(gfr_to_approx('< 26.6175'), 13.30875)
assert np.isnan(gfr_to_approx('≥ 227.944'))
assert np.isnan(gfr_to_approx('p'))

# Original data remains unchanged except for this new proxy column.
df['gfr_approx'] = df['grf'].apply(gfr_to_approx)
print('Regression-eligible rows:', df['gfr_approx'].notna().sum())
print('Excluded only from regression:', df['gfr_approx'].isna().sum())
display(df[['grf', 'gfr_approx']].drop_duplicates().sort_values('gfr_approx'))

In [ ]:
plt.hist(df['gfr_approx'].dropna(), bins=10, edgecolor='black')
plt.title('Distribution of approximate GFR target')
plt.xlabel('Approximate GFR (interval-derived proxy)')
plt.ylabel('Records')
plt.show()

# Part 1 — Regression

## 4. Define regression features and split the data

Drop `class`, `stage`, `affected`, and original `grf` because they are outcome-related or direct target information. The GFR proxy is the target and must never be present among input features.

The train/test split happens **before fitting imputers, scalers, or encoders**.

In [ ]:
reg_df = df.dropna(subset=['gfr_approx']).copy()
regression_drop = ['grf', 'gfr_approx', 'class', 'stage', 'affected']
X_reg = reg_df.drop(columns=regression_drop)
y_reg = reg_df['gfr_approx']

assert not set(regression_drop).intersection(X_reg.columns)

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_STATE
)
print('Regression train/test:', X_reg_train.shape, X_reg_test.shape)
print('Target range:', round(y_reg.min(), 2), 'to', round(y_reg.max(), 2))

## 5. Regression pipeline

Numerical columns receive median imputation and scaling. Object/category columns receive most-frequent imputation and one-hot encoding. The categorical lab ranges are **treated as categories** in this baseline; converting them into clinically meaningful numerical measurements would require a separately validated mapping.

All preprocessing is fitted only on the training subset through the `Pipeline`.

In [ ]:
numeric_reg = X_reg_train.select_dtypes(include='number').columns.tolist()
categorical_reg = X_reg_train.select_dtypes(exclude='number').columns.tolist()

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])

reg_preprocessor = ColumnTransformer([
    ('num', num_pipeline, numeric_reg),
    ('cat', cat_pipeline, categorical_reg),
])
reg_model = Pipeline([
    ('preprocess', reg_preprocessor),
    ('model', LinearRegression()),
])

reg_model.fit(X_reg_train, y_reg_train)
y_reg_pred = reg_model.predict(X_reg_test)
print('Linear Regression trained on', len(X_reg_train), 'rows.')

## 6. Regression evaluation and a simple baseline

In [ ]:
mae = mean_absolute_error(y_reg_test, y_reg_pred)
mse = mean_squared_error(y_reg_test, y_reg_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_reg_test, y_reg_pred)

reg_baseline = DummyRegressor(strategy='median')
reg_baseline.fit(X_reg_train, y_reg_train)
base_reg_pred = reg_baseline.predict(X_reg_test)

print('Linear Regression | MAE:', round(mae, 2),
      '| RMSE:', round(rmse, 2), '| R²:', round(r2, 3))
print('Median baseline   | MAE:', round(mean_absolute_error(y_reg_test, base_reg_pred), 2),
      '| RMSE:', round(np.sqrt(mean_squared_error(y_reg_test, base_reg_pred)), 2))

display(pd.DataFrame({
    'Approximate GFR target': y_reg_test.to_numpy(),
    'Predicted proxy': y_reg_pred,
}).head(10))

### Actual versus predicted

In [ ]:
plt.scatter(y_reg_test, y_reg_pred, alpha=0.7)
low = min(y_reg_test.min(), y_reg_pred.min())
high = max(y_reg_test.max(), y_reg_pred.max())
plt.plot([low, high], [low, high], linestyle='--', color='gray')
plt.xlabel('Actual interval-derived proxy')
plt.ylabel('Predicted proxy')
plt.title('Regression: actual vs. predicted')
plt.show()

### Residuals

In [ ]:
residuals = y_reg_test.to_numpy() - y_reg_pred
plt.scatter(y_reg_pred, residuals, alpha=0.7)
plt.axhline(0, linestyle='--', color='gray')
plt.xlabel('Predicted proxy')
plt.ylabel('Residual (actual proxy - predicted proxy)')
plt.title('Regression residuals')
plt.show()

# Part 2 — Classification

## 7. Prepare classification data from the original complete dataset

**Important correction:** Classification does not require a usable GFR target. We therefore use all available rows with a valid `class` label (including those excluded from regression).

Encode `notckd = 0`, `ckd = 1`. Exclude `grf`, its derived proxy, `stage`, and `affected`, which are closely related to outcome or severity. This helps reduce obvious leakage but **does not prove** the remaining data is leakage-free.

In [ ]:
class_labels = df['class'].astype('string').str.strip().str.lower()
df_clf = df.copy()
df_clf['class_numeric'] = class_labels.map({'notckd': 0, 'ckd': 1})
df_clf = df_clf.dropna(subset=['class_numeric']).copy()
df_clf['class_numeric'] = df_clf['class_numeric'].astype(int)

classification_drop = [
    'class', 'class_numeric', 'grf', 'gfr_approx', 'stage', 'affected'
]
X_clf = df_clf.drop(columns=classification_drop)
y_clf = df_clf['class_numeric']

assert not set(classification_drop).intersection(X_clf.columns)
print('Classification records:', len(X_clf))
print('Classes:', y_clf.value_counts().sort_index().to_dict())

## 8. Stratified train/test split

In [ ]:
X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=RANDOM_STATE,
    stratify=y_clf,
)
print('Classification train/test:', X_clf_train.shape, X_clf_test.shape)
print('Train class ratio:', y_clf_train.value_counts(normalize=True).round(3).to_dict())
print('Test class ratio: ', y_clf_test.value_counts(normalize=True).round(3).to_dict())

## 9. Classification pipeline

The one-hot encoder can ignore categories that are absent in training but appear in testing. All preprocessing remains inside a pipeline.

In [ ]:
numeric_clf = X_clf_train.select_dtypes(include='number').columns.tolist()
categorical_clf = X_clf_train.select_dtypes(exclude='number').columns.tolist()

clf_preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), numeric_clf),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore')),
    ]), categorical_clf),
])

clf_model = Pipeline([
    ('preprocess', clf_preprocessor),
    ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])

## 10. Cross-validation on training data only

Use **five stratified, shuffled folds**. Cross-validation is performed only on the training partition, leaving the held-out test set untouched until final model evaluation.

We also evaluate a small five-feature baseline to see how performance changes with a smaller feature set. This comparison is exploratory and should not be treated as proof of clinical validity.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(
    clf_model, X_clf_train, y_clf_train,
    cv=cv, scoring='accuracy', error_score='raise'
)
print('Full model training CV accuracy:', np.round(cv_scores, 3))
print('Mean:', round(cv_scores.mean(), 3), 'Std:', round(cv_scores.std(), 3))

simple_features = ['htn', 'dm', 'appet', 'pe', 'ane']
simple_model = Pipeline([
    ('scale', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])
simple_scores = cross_val_score(
    simple_model, X_clf_train[simple_features], y_clf_train,
    cv=cv, scoring='accuracy', error_score='raise'
)
print('Five-feature training CV accuracy:', np.round(simple_scores, 3))
print('Mean:', round(simple_scores.mean(), 3))

## 11. Train final model and evaluate on unseen test data

In [ ]:
clf_model.fit(X_clf_train, y_clf_train)
y_clf_pred = clf_model.predict(X_clf_test)

accuracy = accuracy_score(y_clf_test, y_clf_pred)
precision = precision_score(y_clf_test, y_clf_pred, pos_label=1, zero_division=0)
recall = recall_score(y_clf_test, y_clf_pred, pos_label=1, zero_division=0)
f1 = f1_score(y_clf_test, y_clf_pred, pos_label=1, zero_division=0)

base_clf = DummyClassifier(strategy='most_frequent')
base_clf.fit(X_clf_train, y_clf_train)
baseline_accuracy = accuracy_score(y_clf_test, base_clf.predict(X_clf_test))

print('TEST RESULTS (ckd=positive class)')
print('Accuracy: ', round(accuracy, 3))
print('Precision:', round(precision, 3))
print('Recall:   ', round(recall, 3))
print('F1 score: ', round(f1, 3))
print('Majority-class baseline accuracy:', round(baseline_accuracy, 3))
print('Test records:', len(y_clf_test))

### Classification report and confusion matrix

In [ ]:
print(classification_report(
    y_clf_test, y_clf_pred, labels=[0, 1],
    target_names=['Not CKD', 'CKD'], zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_clf_test, y_clf_pred, labels=[0, 1],
    display_labels=['Not CKD', 'CKD'], cmap='Blues', values_format='d'
)
plt.title('CKD classification — held-out test set')
plt.show()

### Predicted CKD probabilities

In [ ]:
# Use the class label to select the correct probability column.
classes = clf_model.named_steps['model'].classes_
ckd_col = list(classes).index(1)
proba_ckd = clf_model.predict_proba(X_clf_test)[:, ckd_col]

display(pd.DataFrame({
    'Actual CKD (1=yes)': y_clf_test.to_numpy(),
    'Predicted CKD (1=yes)': y_clf_pred,
    'Predicted P(CKD)': proba_ckd,
}).head(10))

## 12. Leakage checks and interpretation

- Excluded outcome-related variables `class`, `stage`, `affected`, and `grf` from classification inputs.
- Excluded the regression target and outcome-related variables from regression inputs.
- Fitted all preprocessing within pipelines, so imputation, scaling, and encoding learn **only from training folds**.
- Ran classification cross-validation on the **training set only**.

**Caution:** Very high classification accuracy is not automatically a sign of a great model. This dataset is small, and laboratory attributes may be unusually predictive or closely tied to how labels were assigned. Because patient IDs, recruitment methods, and independent validation data are unavailable, we cannot rule out all sources of leakage, sampling bias, or distribution shift. Test probabilities should not be treated as calibrated clinical risk estimates.

## 13. Results and limitations

After running the notebook, interpret the **new output values** rather than any metrics from an older run.

**Regression:** Linear Regression is a basic baseline on an **interval-derived GFR proxy**. Its performance depends strongly on the assumptions used to turn ranges into numbers. The upper open-ended category is excluded; the lower category uses a non-negative lower bound assumption. Report the regression sample size and compare against the median baseline.

**Classification:** Logistic Regression predicts CKD labels using lab/risk-factor categories. Compare held-out accuracy, precision, recall, F1, confusion matrix, majority-class baseline, and **training-only** CV scores. False negatives matter particularly when considering medical screening, but no threshold tuning or clinical deployment has been validated.

**Data quality:** Some categorical values resemble spreadsheet-formatted dates (`1-Jan`, `3-Mar`), and `grf` contains invalid text (`p`). Without a reliable source codebook, the notebook does not guess their intended clinical values.

**Conclusion:** These pipelines demonstrate preprocessing, leakage-aware design, and evaluation for supervised ML. This educational dataset and workflow are **not appropriate for individual medical decisions**. Additional independent data, verified feature definitions, and clinical validation would be needed before any clinical use.

## 14. Dataset credit

Islam, M. & Akter, S. (2020). *Risk Factor Prediction of Chronic Kidney Disease* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5WP64. Licensed under **CC BY 4.0**.

**To reproduce:** Run all cells from top to bottom in Colab. The dataset is retrieved directly from UCI using `ucimlrepo` (internet connection required). No credentials or private patient files are used.